## Retrieval:
### Similarity Search

In [1]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

from config import OPEN_AI_KEY as API_KEY

embedding = OpenAIEmbeddings(model="text-embedding-3-small", api_key=API_KEY)
vector_store = Chroma(persist_directory="./files", embedding_function=embedding)

In [6]:
question = "What programming languages do data scientists use"
retrieved_docs = vector_store.similarity_search(query=question, k=5)
retrieved_docs

[Document(id='26ffbece-cf4a-40eb-8725-7235e8e0b495', metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science -', 'Course Title': 'Introduction to Splitting'}, page_content='Thus, we need a lot of computational power, and we can expect people to use the languages similar to those in the big data column. Apart from R, Python, and MATLAB, other, faster languages are used like Java, JavaScript, C, C++, and Scala. Cool. What we said may be wonderful, but that’s not all! By using one or more programming languages, people create application software or, as they are sometimes called, software solutions, that are adjusted for specific business needs'),
 Document(id='c022f2d3-f892-4047-968e-11613243f78c', metadata={'Course Title': 'Introduction to Splitting', 'Lecture Title': 'Programming Languages & Software Employed in Data Science -'}, page_content='What about big data? Apart from R and Python, people working in this area are often proficient in other languages l

In [14]:
# Gets popular languages like JS, Python, R, etc. But it's flawed as it can fetch duplicate documents and can also miss relevant chunk of texts.
# Maximal Marginal Relevance (MMR) Search is helpful for this.
question = "What software do data scientists use?"  # Answer should be related to Power BI, SaS, Hadoop etc. not languages
retrieved_docs_1 = vector_store.similarity_search(question, 3)
retrieved_docs_1[0]

Document(id='8865561f-79a7-4692-8f53-e9d86404ef99', metadata={'Course Title': 'Introduction to Splitting', 'Lecture Title': 'Programming Languages & Software Employed in Data Science -'}, page_content='As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end')

In [15]:
retrieved_docs_1[1]

Document(id='89e27b26-4488-4366-9725-db564bc7bbd8', metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science -', 'Course Title': 'Introduction to Splitting'}, page_content='It’s actually a software framework which was designed to address the complexity of big data and its computational intensity. Most notably, Hadoop distributes the computational tasks on multiple computers which is basically the way to handle big data nowadays. Power BI, SaS, Qlik, and especially Tableau are top-notch examples of software designed for business intelligence visualizations')

In [16]:
retrieved_docs_2 = vector_store.max_marginal_relevance_search(question, 3, lambda_mult=1)  # Lambda_mult is between 0(diversity) - 1(relevance)
retrieved_docs_2

[Document(id='8865561f-79a7-4692-8f53-e9d86404ef99', metadata={'Course Title': 'Introduction to Splitting', 'Lecture Title': 'Programming Languages & Software Employed in Data Science -'}, page_content='As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end'),
 Document(id='89e27b26-4488-4366-9725-db564bc7bbd8', metadata={'Course Title': 'Introduction to Splitting', 'Lecture Title': 'Programming Languages & Software Employed in Data Science -'}, page_content='It’s actually a software framework which was designed to address the complexity of big data and its computational intensity. Most notably, Had

In [19]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k":3, "lambda_mult": 0.7})
retrieved_docs = retriever.invoke(question)
retrieved_docs

[Document(id='8865561f-79a7-4692-8f53-e9d86404ef99', metadata={'Course Title': 'Introduction to Splitting', 'Lecture Title': 'Programming Languages & Software Employed in Data Science -'}, page_content='As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end'),
 Document(id='89e27b26-4488-4366-9725-db564bc7bbd8', metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science -', 'Course Title': 'Introduction to Splitting'}, page_content='It’s actually a software framework which was designed to address the complexity of big data and its computational intensity. Most notably, Had

### Generation: Stuffing Documents

In [54]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

str_parser = StrOutputParser()
pass_through = RunnablePassthrough()
chat = ChatOpenAI(model="gpt-5.6-luna", seed=365, temperature=0, api_key=API_KEY)
template = """
Answer the following question: {question}

To answer the question, use only the following context:
{context}

At the end of the response, please specify the name of the lecture this context is taken from in the format:
Resources: *Lecture Title* where *Lecture Title* should be substituted with the title of all resource lectures.
"""

prompt_template = PromptTemplate.from_template(template)
chain = {"context": retriever, "question": pass_through} | prompt_template | chat

In [41]:
chain.invoke(question)

{'context': [Document(id='8865561f-79a7-4692-8f53-e9d86404ef99', metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science -', 'Course Title': 'Introduction to Splitting'}, page_content='As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end'),
  Document(id='89e27b26-4488-4366-9725-db564bc7bbd8', metadata={'Course Title': 'Introduction to Splitting', 'Lecture Title': 'Programming Languages & Software Employed in Data Science -'}, page_content='It’s actually a software framework which was designed to address the complexity of big data and its computational intensity. Most

In [55]:
result = chain.invoke(question)

In [62]:
print(result.text)

Data scientists use:

- **R and Python** for data manipulation, statistical computations, and solving data-related problems.
- **Hadoop** for processing and distributing big-data computations across multiple computers.
- **Power BI, SAS, Qlik, and Tableau** for business-intelligence visualizations.

Resources: *Programming Languages & Software Employed in Data Science -*
